# JJMO Flux Calibration Pipeline -- Quickstart

End-to-end run of the `jjmo_fluxcal` pipeline on JJMO Sirius data, from raw
segmented spectra to a flux-calibrated spectrum in physical units.

**Workflow:** read -> wavelength calibrate -> quality assess -> derive
sensitivity from the per-segment counts -> flux-calibrate each segment ->
stitch the calibrated segments. Calibrating before stitching corrects each
segment's wavelength-dependent throughput, so the segments line up much
better in the final stitch.

Run cells top-to-bottom; the full notebook completes in well under a
second of compute time (plus a few seconds of one-time imports the first
time you run cell 2).

**Environment:** the `jjmo_spec` conda env (specutils >= 2.3, astropy >= 7).


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Spectrum1D.*")

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

import jjmo_fluxcal
from jjmo_fluxcal import io, wavelength, quality, stitching, reference, sensitivity, calibrate
from jjmo_fluxcal.config import PipelineConfig

# specutils 2.3 renamed Spectrum1D -> Spectrum; alias keeps the old name available
try:
    from specutils import Spectrum as Spectrum1D
except ImportError:
    from specutils import Spectrum1D

print(f"jjmo_fluxcal v{jjmo_fluxcal.__version__}")

## Configuration

`PipelineConfig` holds every knob the pipeline exposes. Defaults are fine
for Sirius; override fields to customise.

In [ ]:
cfg = PipelineConfig()
print(f"Star:        {cfg.star_name}")
print(f"Fit method:  {cfg.fit_method}, order {cfg.fit_order}")
print(f"Sigma clip:  {cfg.sigma_clip}")
print(f"Mask Balmer: {cfg.mask_balmer}, telluric: {cfg.mask_telluric}")

## Step 1 -- Data Ingestion

`io.read_directory` auto-detects file formats (.fit/.txt for Sirius, .csv
for Betelgeuse) and returns a list of `Spectrum1D` objects (one per
~500 A segment), already sorted by ascending wavelength.

In [ ]:
SIRIUS_DIR = "/home/habjan.e/JJMO_home/Data/Sirius"

segments = io.read_directory(SIRIUS_DIR)
print(f"Loaded {len(segments)} segments")

for seg in segments:
    w = seg.spectral_axis.to(u.AA).value
    print(f"  {seg.meta.get('segment_id', '?'):>8s}: {w[0]:.0f} - {w[-1]:.0f} A  ({len(w)} px)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for seg in segments:
    w = seg.spectral_axis.to(u.AA).value
    f = seg.flux.value
    ax.plot(w, f, lw=0.8, label=seg.meta.get("segment_id", ""))
ax.set_xlabel("Wavelength (A)")
ax.set_ylabel("Counts")
ax.set_title("Raw Sirius Segments")
ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

## Step 2 -- Wavelength Calibration

Detect absorption lines, match to known rest wavelengths, measure the
instrumental offset (from telluric lines) and radial velocity (from stellar
lines), then correct the wavelength arrays.

In [ ]:
wavelengths = [seg.spectral_axis.to(u.AA).value for seg in segments]
fluxes      = [seg.flux.value for seg in segments]
seg_ids     = [seg.meta.get("segment_id", f"seg_{i:02d}") for i, seg in enumerate(segments)]

solutions = wavelength.calibrate_segments(wavelengths, fluxes, segment_ids=seg_ids)
wavelength.print_calibration_table(solutions)

# Wavelength arrays after offset correction (observatory frame)
corrected_w = [s.wavelength_observatory for s in solutions]

## Step 3 -- Quality Assessment

Per-segment checks for edge vignetting, cosmic rays, telluric contamination,
and SNR. The returned `QualityReport.mask_good` is `True` for usable pixels.

In [ ]:
reports = quality.assess_segments(corrected_w, fluxes, segment_ids=seg_ids)
quality.print_quality_table(reports)

# Build Spectrum1D segments with corrected wavelengths and quality masks --
# these are reused by the sensitivity, calibration, and stitching steps below.
corrected_segments = [
    Spectrum1D(
        spectral_axis=w * u.AA,
        flux=f * u.ct,
        mask=~qr.mask_good,            # specutils convention: True = bad
        meta={"segment_id": seg_ids[i]},
    )
    for i, (w, f, qr) in enumerate(zip(corrected_w, fluxes, reports))
]

## Step 4 -- Reference Spectrum

Load the CALSPEC reference for Sirius. (Cached locally after the first
download, so subsequent loads are instant.)

In [ ]:
ref_spec = reference.load_reference_spectrum("sirius", prefer="calspec")
ref_w = ref_spec.spectral_axis.to(u.AA).value
ref_f = ref_spec.flux.value
print(f"Reference: {ref_w[0]:.0f} - {ref_w[-1]:.0f} A, {len(ref_w)} points")

## Step 5 -- Sensitivity Function

Derive `S(lambda) = F_ref / C_obs` from the per-segment counts, fit a smooth
low-order Chebyshev (masking stellar/telluric features), and combine the
per-segment fits into a single global sensitivity function.

The sensitivity step works on the **per-segment counts**, not on a stitched
spectrum, so it doesn't matter that we haven't stitched yet.

In [ ]:
sens_global = sensitivity.derive_sensitivity(
    segments_obs   = [(w, f) for w, f in zip(corrected_w, fluxes)],
    wavelength_ref = ref_w,
    flux_ref       = ref_f,
    masks          = [qr.mask_good for qr in reports],
    segment_ids    = seg_ids,
    fit_method     = "chebyshev",
    fit_order      = 4,
    sigma_clip     = 3.0,
)

n_seg = len(sens_global.segment_fits)
gfit  = sens_global.global_fit
print(f"Per-segment fits: {n_seg}")
if gfit is not None:
    print(f"Global fit:       {gfit.method} order {gfit.order}, "
          f"RMS residual = {gfit.rms_residual:.3g}, "
          f"used {gfit.n_points_used} points (rejected {gfit.n_rejected})")

## Step 6 -- Flux Calibration (per segment)

Apply the global sensitivity function to **each segment** before stitching.
This removes the wavelength-dependent throughput within every segment, so
the segments line up much better at their overlap regions than they would
in raw counts.

`GlobalSensitivity.to_sensitivity_function(...)` evaluates the global fit
on a wavelength grid and returns a `calibrate.SensitivityFunction` in the
convention `apply_sensitivity` expects. We pass `exptime=1` because the
segment fluxes are already per-pixel counts.

In [ ]:
# Sample the sensitivity function on a dense grid spanning all segments
grid_min = min(w.min() for w in corrected_w)
grid_max = max(w.max() for w in corrected_w)
sens_func = sens_global.to_sensitivity_function(
    np.linspace(grid_min, grid_max, 4000),
    meta={"star": "sirius", "fit_method": "chebyshev", "fit_order": 4},
)

# Apply per-segment to get a list of CalibrationResult, then convert each
# back to Spectrum1D so the stitcher can ingest them.
cal_results  = calibrate.apply_sensitivity_per_segment(
    corrected_segments, sens_func, exptime=1.0,
)
cal_segments = [r.to_spectrum1d() for r in cal_results]

print(f"Calibrated {len(cal_segments)} segments")
print(f"Flux unit:  {cal_segments[0].flux.unit}")

In [ ]:
# Plot the per-segment calibrated spectra (before stitching)
fig, ax = plt.subplots(figsize=(12, 4))
for sp in cal_segments:
    w = sp.spectral_axis.to(u.AA).value
    f = sp.flux.value
    good = ~sp.mask if sp.mask is not None else slice(None)
    ax.plot(w[good], f[good], lw=0.7, alpha=0.85,
            label=sp.meta.get("segment_id", ""))

# CALSPEC reference overlay, trimmed to observed range
m = (ref_w >= grid_min) & (ref_w <= grid_max)
ax.plot(ref_w[m], ref_f[m], "k--", lw=0.6, alpha=0.6, label="CALSPEC")

ax.set_xlabel("Wavelength (A)")
ax.set_ylabel(r"Flux (erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")
ax.set_title("Flux-calibrated segments (before stitching)")
ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

## Step 7 -- Stitching the Calibrated Segments

With each segment already on a common physical-flux scale, stitch them
into a single continuous spectrum. We pass `mode='post_calibration'`,
which skips cross-normalization unless adjacent segments still disagree
by more than 10% in their overlap regions.

In [ ]:
stitched = stitching.stitch_segments(
    cal_segments,
    mode="post_calibration",
    normalize=True,
    resample_method="interp",
)

print(f"Stitched: {stitched.wavelength[0]:.0f} - {stitched.wavelength[-1]:.0f} A "
      f"({len(stitched.wavelength)} px)")
print(f"Median flux: {np.nanmedian(stitched.flux):.3e} erg/s/cm^2/A")

In [ ]:
# Final spectrum vs CALSPEC reference
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(stitched.wavelength, stitched.flux, "k-", lw=0.6,
        label="Stitched (JJMO, calibrated)", alpha=0.85)
if stitched.uncertainty is not None:
    ax.fill_between(stitched.wavelength,
                    stitched.flux - stitched.uncertainty,
                    stitched.flux + stitched.uncertainty,
                    alpha=0.15, color="gray")

# CALSPEC reference, trimmed to observed range
wmin, wmax = stitched.wavelength[0], stitched.wavelength[-1]
m = (ref_w >= wmin) & (ref_w <= wmax)
ax.plot(ref_w[m], ref_f[m], "r-", lw=0.6, alpha=0.7, label="CALSPEC reference")

ax.set_xlabel("Wavelength (A)")
ax.set_ylabel(r"Flux (erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")
ax.set_title("Sirius: final calibrated, stitched spectrum vs CALSPEC")
ax.legend()
plt.tight_layout()
plt.show()

## One-call pipeline

`jjmo_fluxcal.fluxcal` is a single-call entry point that runs the full
pipeline. Note: its current implementation passes outdated keyword names
to the sensitivity step and stitches before calibrating; until that's
reconciled, prefer the cell-by-cell flow above.

```python
from jjmo_fluxcal import fluxcal

result = fluxcal(
    "/home/habjan.e/JJMO_home/Data/Sirius",
    star_name="sirius",
    output_dir="./results",
    fit_order=4,
    sigma_clip=3.0,
)
# result["calibrated"], result["sensitivity"], result["stitched"], ...
```

## CLI usage

```bash
# Full pipeline
jjmo-fluxcal run --input-dir ./data/Sirius --star sirius --output-dir ./results

# Derive a sensitivity function only
jjmo-fluxcal sensfunc --input-dir ./data/Sirius --star sirius --output sensfunc.json

# Apply a saved sensitivity function to new science data
jjmo-fluxcal apply --sensfunc sensfunc.json --input science.fits --output calibrated.fits
```

Use `--config my_config.yaml` to load parameters from a YAML file; example
configs live in `jjmo_fluxcal/examples/`.